# **RUN THE COMMAND 2 TIMES**

In [1]:
!pip install mediapipe opencv-python tensorflow

# **BASIC MODULES**

In [1]:
import pandas as pd
import numpy as np
import os
import logging

# ***ESSENTIAL MEDIAPIPE IMPORTS***

In [3]:
import cv2
import mediapipe as mp
from mediapipe.framework.formats import landmark_pb2
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# **Drive Mount**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
DATA_PATH = "/content/drive/MyDrive/Merged"

# **Class Labels**

In [7]:
np.random.seed(42)
# Parameters
# (x,y,z,visibility) -> for pose
# (x,y,z) -> for hands
MAX_SEQUENCE_LENGTH = 75
NUM_FEATURES = 195
CLASSES = ['Afternoon', 'Apple', 'April', 'August', 'Banana', 'Day', 'December', 'Evening',
           'Febraury', 'Friday', 'Grapes', 'January', 'July', 'June', 'March', 'May', 'Monday',
           'Morning', 'Night', 'November', 'October', 'Orange', 'Rainy', 'Saturday', 'September',
           'Summer', 'Sunday', 'Thursday', 'Tuesday', 'Valencia_Orange', 'Watermelon', 'Wednesday', 'Winter']

NUM = len(CLASSES)

print(NUM)
print(NUM_FEATURES)

33
195


# **RUN THE BELOW CELLS FOR FEATURE EXTRACTION**

In [12]:
def extract_mediapipe_features(video_path):
    sequence_features = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return None

    desired_pose_landmark_indices = set(range(23))

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Convert the BGR image to RGB.
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False # To improve performance

            # Process the image and find landmarks.
            results = holistic.process(image)

            # Revert to BGR and enable writing for drawing
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            # --- Extract only desired features ---
            current_frame_features = []

            if results.pose_landmarks:
                for i, landmark in enumerate(results.pose_landmarks.landmark):
                    if i in desired_pose_landmark_indices:
                        current_frame_features.extend([landmark.x, landmark.y, landmark.z])

                num_expected_pose_features = len(desired_pose_landmark_indices) * 3
                if len(current_frame_features) < num_expected_pose_features:

                    current_frame_features.extend([0.0] * (num_expected_pose_features - len(current_frame_features)))
            else:
                current_frame_features.extend([0.0] * len(desired_pose_landmark_indices) * 3)

            if results.left_hand_landmarks:
                for landmark in results.left_hand_landmarks.landmark:
                    current_frame_features.extend([landmark.x, landmark.y, landmark.z])
            else:
                current_frame_features.extend([0.0] * 21 * 3)

            if results.right_hand_landmarks:
                for landmark in results.right_hand_landmarks.landmark:
                    current_frame_features.extend([landmark.x, landmark.y, landmark.z])
            else:
                current_frame_features.extend([0.0] * 21 * 3) # 63 zeros

            # Verify the total number of features matches NUM_FEATURES
            if len(current_frame_features) != NUM_FEATURES:
                print(f"Warning: Feature count mismatch at frame {frame_count}. Expected {NUM_FEATURES}, got {len(current_frame_features)}")
                # Fallback to ensure consistent shape by padding with zeros if mismatch occurs
                if len(current_frame_features) < NUM_FEATURES:
                     current_frame_features.extend([0.0] * (NUM_FEATURES - len(current_frame_features)))
                else: # Truncate if too many (unlikely with this logic but good practice)
                    current_frame_features = current_frame_features[:NUM_FEATURES]


            sequence_features.append(current_frame_features)
            frame_count += 1

    cap.release()

    if not sequence_features:
        print(f"Warning: No features extracted from {video_path}")
        return np.zeros((0, NUM_FEATURES)) # Return empty array with correct feature dimension

    return np.array(sequence_features)


def standardize_sequence(sequence, max_len, num_features):
    """Pads or truncates a sequence to a fixed length."""
    if len(sequence) > max_len:
        # Truncate the sequence
        return sequence[:max_len]
    elif len(sequence) < max_len:
        # Pad the sequence with zeros
        padding = np.zeros((max_len - len(sequence), num_features))
        return np.vstack((sequence, padding))
    else:
        # Sequence is already the correct length
        return sequence

In [13]:
# --- 1. Load Data (Video Paths and Labels) ---
def load_data(data_path, classes_list, max_seq_length, num_features_per_frame):
    sequences = []
    labels = []
    label_map = {label: num for num, label in enumerate(classes_list)}

    for class_name in classes_list:
        class_path = os.path.join(data_path, class_name)
        if not os.path.isdir(class_path):
            print(f"Warning: Directory not found for class {class_name} at {class_path}")
            continue

        print(f"Processing class: {class_name}")
        video_count = 0
        for video_file in os.listdir(class_path):
            video_path = os.path.join(class_path, video_file)
            # Basic check for video file extensions, add more if needed
            if not (video_file.lower().endswith('.mp4') or \
                    video_file.lower().endswith('.avi') or \
                    video_file.lower().endswith('.mov')):
                print(f"Skipping non-video file: {video_file} in {class_name}")
                continue

            # Set display_video=True for debugging a single video, False for batch processing
            keypoints = extract_mediapipe_features(video_path)

            if keypoints is not None and keypoints.shape[0] > 0:
                # Preprocess: pad or truncate
                processed_keypoints = standardize_sequence(keypoints, max_len=max_seq_length, num_features=num_features_per_frame)
                sequences.append(processed_keypoints)
                labels.append(label_map[class_name])
                video_count +=1
            else:
                print(f"Warning: Could not extract features or no frames from {video_path}. Skipping.")
        print(f"Processed {video_count} videos for class {class_name}")


    if not sequences:
        print("Error: No sequences were loaded. Check DATA_PATH and video files.")
        return None, None

    return np.array(sequences), np.array(labels)

In [14]:
X, y = load_data(DATA_PATH, CLASSES, MAX_SEQUENCE_LENGTH, NUM_FEATURES)

Processing class: Afternoon
Processed 64 videos for class Afternoon
Processing class: Apple
Processed 64 videos for class Apple
Processing class: April
Processed 70 videos for class April
Processing class: August
Processed 68 videos for class August
Processing class: Banana
Processed 63 videos for class Banana
Processing class: Day
Processed 68 videos for class Day
Processing class: December
Processed 72 videos for class December
Processing class: Evening
Processed 67 videos for class Evening
Processing class: Febraury
Processed 71 videos for class Febraury
Processing class: Friday
Processed 70 videos for class Friday
Processing class: Grapes
Processed 73 videos for class Grapes
Processing class: January
Processed 71 videos for class January
Processing class: July
Processed 72 videos for class July
Processing class: June
Processed 70 videos for class June
Processing class: March
Processed 69 videos for class March
Processing class: May
Processed 67 videos for class May
Processing class

# **To save the extracted sequences of frames**

In [15]:
# prompt: save X and y in npy format

np.save('X.npy', X)
np.save('y.npy', y)
print("X and y saved successfully.")

X and y saved successfully.


In [18]:
# prompt: Iwanna save X and y in that dir

np.save('/content/drive/MyDrive/Frames/X_final_conference.npy', X)
np.save('/content/drive/MyDrive/Frames/y_final_conference.npy', y)

print("X and y saved successfully to Google Drive.")

X and y saved successfully to Google Drive.


In [4]:
# prompt: Fetch X and y from the drive

X = np.load('/content/drive/MyDrive/Frames/X_final_conference.npy')
y = np.load('/content/drive/MyDrive/Frames/y_final_conference.npy')

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (2319, 75, 195)
Shape of y: (2319,)
